Exemple complet en PyTorch (série multivariée + LSTM + GRU)

Ici :

Entrées : consommation, température, humidité

Sortie : consommation future

In [1]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:

# --------------------------------
# 1. Génération de données multivariées factices
# --------------------------------
np.random.seed(0)
n = 1000
t = np.arange(n)

consumption = 50 + 10*np.sin(t/50) + np.random.randn(n)
temperature = 20 + 5*np.sin(t/100) + 0.5*np.random.randn(n)
humidity = 60 + 10*np.cos(t/80) + np.random.randn(n)

# Stack : (n, 3 variables)
data = np.column_stack((consumption, temperature, humidity))

# Normalisation multivariée
scaler = MinMaxScaler()
data = scaler.fit_transform(data)

# --------------------------------
# 2. Création des fenêtres temporelles
# --------------------------------
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length, :])   # toutes les variables
        y.append(data[i+seq_length, 0])     # prédire seulement la consommation
    return np.array(X), np.array(y)

seq_length = 20
X, y = create_sequences(data, seq_length)

# Split train / test
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Tensors PyTorch
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train).unsqueeze(1)
X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test).unsqueeze(1)

print("Shape X :", X_train.shape)
# (batch, seq_length, nb_variables) → (batch, 20, 3)

# --------------------------------
# 3. Modèle LSTM multivarié
# --------------------------------
class LSTMModel(nn.Module):
    def __init__(self, input_size=3, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]      # dernier pas de temps
        out = self.fc(out)
        return out

# --------------------------------
# 4. Modèle GRU multivarié
# --------------------------------
class GRUModel(nn.Module):
    def __init__(self, input_size=3, hidden_size=64, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

# --------------------------------
# 5. Entraînement générique
# --------------------------------
def train_model(model, X_train, y_train, epochs=20, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

        if (epoch+1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# --------------------------------
# 6. Évaluation
# --------------------------------
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        preds = model(X_test).numpy()
        y_true = y_test.numpy()

    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mae = mean_absolute_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100
    return rmse, mae, mape

# --------------------------------
# 7. LSTM
# --------------------------------
print("===== LSTM multivarié =====")
lstm = LSTMModel(input_size=3)
train_model(lstm, X_train, y_train)

rmse, mae, mape = evaluate_model(lstm, X_test, y_test)
print(f"LSTM -> RMSE: {rmse:.3f}, MAE: {mae:.3f}, MAPE: {mape:.2f}%")

# --------------------------------
# 8. GRU
# --------------------------------
print("\n===== GRU multivarié =====")
gru = GRUModel(input_size=3)
train_model(gru, X_train, y_train)

rmse, mae, mape = evaluate_model(gru, X_test, y_test)
print(f"GRU -> RMSE: {rmse:.3f}, MAE: {mae:.3f}, MAPE: {mape:.2f}%")


Shape X : torch.Size([784, 20, 3])
===== LSTM multivarié =====
Epoch [5/20], Loss: 0.2230
Epoch [10/20], Loss: 0.1010
Epoch [15/20], Loss: 0.0953
Epoch [20/20], Loss: 0.0639
LSTM -> RMSE: 0.290, MAE: 0.261, MAPE: 180.29%

===== GRU multivarié =====
Epoch [5/20], Loss: 0.2029
Epoch [10/20], Loss: 0.0568
Epoch [15/20], Loss: 0.0848
Epoch [20/20], Loss: 0.0457
GRU -> RMSE: 0.241, MAE: 0.217, MAPE: 150.12%
